In [ ]:
import geopandas as gpd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import folium

# 1. Preparação rápida dos dados (necessário pois os notebooks não conversam entre si)
municipios = gpd.read_file('dados/municipios-mg.geojson')
focos = gpd.read_file('dados/focos-desmatamento-mg.geojson')
socioeco = pd.read_csv('dados/populacao-pib-municipios-mg.csv')

focos['area_desmatada_ha'] = focos.geometry.area / 10000
focos['area_desmatada_km2'] = focos.geometry.area / 10**6
focos_muni = gpd.sjoin(focos, municipios, how="inner", predicate="intersects")

area_mes_ha = focos.groupby('mes')['area_desmatada_ha'].sum().reset_index()
area_bioma_km2 = focos.groupby('bioma')['area_desmatada_km2'].sum().reset_index()
area_muni_mes_km2 = focos_muni.groupby(['name_muni', 'mes'])['area_desmatada_km2'].sum().reset_index()

desmatamento_total_muni = focos_muni.groupby('name_muni')['area_desmatada_ha'].sum().reset_index()
df_correlacao = pd.merge(desmatamento_total_muni, socioeco, left_on='name_muni', right_on='municipio', how='inner')

# 2. Configurações Visuais
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'sans-serif'

# VISUALIZAÇÕES 1 E 2: Biomas e Tempo
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=area_bioma_km2, x='bioma', y='area_desmatada_km2', ax=axes[0], palette='Reds_r')
axes[0].set_title('Impacto por Bioma', fontsize=14, fontweight='bold')

sns.barplot(data=area_mes_ha, x='mes', y='area_desmatada_ha', ax=axes[1], palette='Oranges_r')
axes[1].set_title('Evolução do Desmatamento', fontsize=14, fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()

# VISUALIZAÇÃO 3: Top 10 Municípios
top10_muni = area_muni_mes_km2.groupby('name_muni')['area_desmatada_km2'].sum().sort_values(ascending=False).head(10).reset_index()
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=top10_muni, y='name_muni', x='area_desmatada_km2', palette='dark:red_r')
plt.title('Os 10 Municípios com Maior Área Desmatada', fontsize=16, fontweight='bold')
plt.xlabel('Área Desmatada (km²)')
plt.ylabel('')
for i in ax.containers: ax.bar_label(i, padding=5, fmt='%.1f km²')
sns.despine(left=True)
plt.show()

# VISUALIZAÇÃO 4: Relação Econômica
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_correlacao, x='pib_mil_reais', y='area_desmatada_ha',
                size='populacao_2022', sizes=(50, 800), alpha=0.6, color='brown')
plt.title('Relação Econômica: PIB vs. Desmatamento', fontsize=14, fontweight='bold')
plt.xlabel('PIB Municipal (Mil Reais)')
plt.ylabel('Desmatamento (Hectares)')
plt.xscale('log')
plt.legend(title='População', bbox_to_anchor=(1.05, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.show()

# VISUALIZAÇÃO 5: Mapa Interativo
mapa_dados = focos_muni.groupby('name_muni')['area_desmatada_km2'].sum().reset_index()
mun_mapa = municipios.merge(mapa_dados, on='name_muni', how='left').fillna(0)
mun_mapa_wgs84 = mun_mapa.to_crs(epsg=4326)

m = folium.Map(location=[-18.5122, -44.5550], zoom_start=6, tiles='CartoDB positron')
folium.Choropleth(
    geo_data=mun_mapa_wgs84,
    name='Áreas Críticas',
    data=mun_mapa_wgs84,
    columns=['name_muni', 'area_desmatada_km2'],
    key_on='feature.properties.name_muni',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Área Desmatada (km²)',
).add_to(m)

m.save('dados/mapa_fiscalizacao.html')
display(m)